In [9]:
import sys
sys.path.append("..")
import numpy as np
import pickle

from lm_conf.default_utils.custom_types import OrganisedOutputs
from lm_conf.post_processing.metrics import BetaDistribution
from lm_conf.post_processing.metrics import dAUROC, dECE_equal_mass, dECE_equal_width, dAUROC_scalar, dECE_scalar

In [2]:
path = """
results/mmlu/dist_lnll/meta-llama/Llama-3.1-8B-Instruct/2025-12-24_12-26-39
""".strip()
with open(f"../{path}/graded_outputs_0.pkl", "rb") as f:
    graded_outputs: OrganisedOutputs = pickle.load(f)

In [3]:
conf: list[BetaDistribution] = graded_outputs.extracted_confidences[0]
acc: list = graded_outputs.accuracy_scores[0]
print(len(conf), len(acc))

14042 14042


In [16]:
# train test split with stratification by confidence mean (mu)
from sklearn.model_selection import train_test_split

mu_values = np.array([c.mu for c in conf])
# bucketize mu into bins to stratify; adjust bin count if data is sparse
num_strata_bins = 10
strata = np.digitize(mu_values, bins=np.linspace(0.0, 1.0, num_strata_bins + 1), right=True)

conf_train, conf_test, acc_train, acc_test, strata_train, strata_test = train_test_split(
    conf,
    acc,
    strata,
    test_size=0.80,
    random_state=42,
    stratify=strata,
) 

print("conf", len(conf_train), len(conf_test))
print("acc", len(acc_train), len(acc_test))
print("strata train/test counts", np.bincount(strata_train), np.bincount(strata_test))

conf 2808 11234
acc 2808 11234
strata train/test counts [  0   0   0 105 362 400 303 235 356 700 347] [   0    0    2  422 1450 1598 1210  938 1426 2799 1389]


In [5]:
def build_outputs(conf_list, acc_list):
    return OrganisedOutputs(
        extracted_confidences=[conf_list],
        accuracy_scores=[acc_list],
    )

In [ ]:
# print("Uncalibrated Test dAUROC", dAUROC({}, build_outputs(conf_test, acc_test)))
# print("Uncalibrated Train dAUROC", dAUROC({}, build_outputs(conf_train, acc_train)))
# print("Uncalibrated Test dAUROC scalar", dAUROC_scalar({}, build_outputs(conf_test, acc_test)))
# print("Uncalibrated Train dAUROC scalar", dAUROC_scalar({}, build_outputs(conf_train, acc_train)))

[2025-12-26 01:06:31] INFO metrics.py:291: Computing dAUROC
Computing dAUROC with global Monte Carlo sampling: 100%|██████████| 20/20 [00:57<00:00,  2.85s/it]
[2025-12-26 01:07:28] INFO metrics.py:291: Computing dAUROC


Uncalibrated Test dAUROC [0.710463]


Computing dAUROC with global Monte Carlo sampling: 100%|██████████| 20/20 [00:56<00:00,  2.83s/it]

Uncalibrated Train dAUROC [0.705856]


In [17]:
print("Uncalibrated Test dECE", dECE_equal_width({"results_path": "."}, build_outputs(conf_test, acc_test)))
print("Uncalibrated Train dECE", dECE_equal_width({"results_path": "."}, build_outputs(conf_train, acc_train)))

[2025-12-26 01:26:57] INFO metrics.py:266: Computing dECE_equal_width


[2025-12-26 01:27:04] INFO metrics.py:260: Saved dECE distribution plots to ./dECE_equal_width_distributions_2.png
[2025-12-26 01:27:04] INFO metrics.py:266: Computing dECE_equal_width


Uncalibrated Test dECE [0.13972195347920435]


[2025-12-26 01:27:07] INFO metrics.py:260: Saved dECE distribution plots to ./dECE_equal_width_distributions_3.png


Uncalibrated Train dECE [0.1294062801513055]


In [ ]:
from scipy.stats import beta
from sklearn.ensemble import RandomForestRegressor

# Step 1: Bin confidence scores into equal-width bins
num_bins = 10
bin_edges = np.linspace(0.0, 1.0, num_bins + 1)

# Dictionary to store beta distributions per bin
bin_beta_dists = {}
bin_accuracies = {}

# Bin the training data
for i in range(num_bins):
    lo, hi = bin_edges[i], bin_edges[i + 1]
    
    # Include right edge for last bin
    if i == num_bins - 1:
        mask = np.array([(c.mu >= lo and c.mu <= hi) for c in conf_train])
    else:
        mask = np.array([(c.mu >= lo and c.mu < hi) for c in conf_train])
    
    bin_accs = np.array([float(acc) for acc, m in zip(acc_train, mask) if m])
    
    if len(bin_accs) > 0:
        # Fit beta distribution to bin's accuracy scores
        # Using method of moments: alpha = mu*(mu*(1-mu)/sigma^2 - 1), beta = (1-mu)*(mu*(1-mu)/sigma^2 - 1)
        bin_mu = np.mean(bin_accs)
        bin_var = np.var(bin_accs)
        
        if bin_var > 0:
            # Ensure parameters are positive
            common = bin_mu * (1 - bin_mu) / (bin_var + 1e-8) - 1
            alpha = max(0.1, bin_mu * common)
            beta_param = max(0.1, (1 - bin_mu) * common)
        else:
            # If no variance, use simple estimate
            alpha = 1.0 + bin_mu * 10
            beta_param = 1.0 + (1 - bin_mu) * 10
        
        bin_beta_dists[i] = {"alpha": alpha, "beta": beta_param, "bin_range": (lo, hi)}
        bin_accuracies[i] = bin_accs
        
        print(f"Bin {i} [{lo:.2f}, {hi:.2f}]: alpha={alpha:.3f}, beta={beta_param:.3f}, n={len(bin_accs)}, acc_mean={bin_mu:.3f}")
    else:
        bin_beta_dists[i] = None
        bin_accuracies[i] = None
        print(f"Bin {i} [{lo:.2f}, {hi:.2f}]: empty")

print("\nBin statistics completed")

Bin 0 [0.00, 0.10]: empty
Bin 1 [0.10, 0.20]: empty
Bin 2 [0.20, 0.30]: alpha=0.100, beta=0.100, n=105, acc_mean=0.305
Bin 3 [0.30, 0.40]: alpha=0.100, beta=0.100, n=362, acc_mean=0.337
Bin 4 [0.40, 0.50]: alpha=0.100, beta=0.100, n=400, acc_mean=0.378
Bin 5 [0.50, 0.60]: alpha=0.100, beta=0.100, n=303, acc_mean=0.505
Bin 6 [0.60, 0.70]: alpha=0.100, beta=0.100, n=235, acc_mean=0.617
Bin 7 [0.70, 0.80]: alpha=0.100, beta=0.100, n=356, acc_mean=0.801
Bin 8 [0.80, 0.90]: alpha=0.100, beta=0.100, n=700, acc_mean=0.879
Bin 9 [0.90, 1.00]: alpha=0.100, beta=0.100, n=347, acc_mean=0.934

Bin statistics completed
